# 17. 날씨 조건부 택시 수요 탄력성 분석

**목적:** 기온/강수량 등 기상 조건 변화에 따른 택시 수요의 탄력성(elasticity)을 분석한다.

**분석 내용:**
- 기온 구간별, 강수량 구간별 택시 수요 변화율 산출
- 기온 x 강수량 x 시간대 3차원 히트맵
- 지역별(행정동) 날씨 민감도 차이 분석
- 탄력성 계수 = (수요변화율) / (기상변화율)

**데이터:** DC_TBYXD012.csv + weather_daily + weather_hourly + calendar

In [ ]:
# 필요 라이브러리 설치
!pip install psutil plotly -q

In [ ]:
# 메모리 모니터링 유틸 + 기본 설정
import psutil
import os
import gc
import warnings
warnings.filterwarnings('ignore')

def print_mem():
    proc = psutil.Process(os.getpid())
    mem = proc.memory_info().rss / 1024**2
    print(f'현재 메모리 사용량: {mem:.0f} MB')

print_mem()

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
from matplotlib import font_manager
import plotly.express as px
import plotly.graph_objects as go

# 한글 폰트 설정
import platform
if platform.system() == 'Windows':
    plt.rcParams['font.family'] = 'Malgun Gothic'
elif platform.system() == 'Darwin':
    plt.rcParams['font.family'] = 'AppleGothic'
else:
    plt.rcParams['font.family'] = 'NanumGothic'
plt.rcParams['axes.unicode_minus'] = False

# 경로 설정
DATA_DIR = './'
EXT_DIR = './external_data/'
TAXI_FILE = os.path.join(DATA_DIR, 'DC_TBYXD012.csv')

print('설정 완료')
print_mem()

## 1. 외부 데이터 로드

In [ ]:
# 일별 기상 데이터
weather_daily = pd.read_csv(
    os.path.join(EXT_DIR, 'weather_asos_daily_seoul_2018_2026.csv'),
    encoding='utf-8',
    usecols=['date', 'avg_temp', 'min_temp', 'max_temp', 'rainfall',
             'avg_wind_speed', 'avg_humidity', 'sunshine_hours'],
    dtype={'avg_temp': 'float32', 'min_temp': 'float32', 'max_temp': 'float32',
           'rainfall': 'float32', 'avg_wind_speed': 'float32',
           'avg_humidity': 'float32', 'sunshine_hours': 'float32'}
)
weather_daily['date'] = pd.to_datetime(weather_daily['date'])
weather_daily['rainfall'] = weather_daily['rainfall'].fillna(0)
print(f'weather_daily: {len(weather_daily)}행')

# 시간별 기상 데이터
weather_hourly = pd.read_csv(
    os.path.join(EXT_DIR, 'weather_asos_hourly_seoul_2018_2026.csv'),
    encoding='utf-8',
    usecols=['date', 'hour', 'temp', 'rainfall', 'humidity'],
    dtype={'hour': 'str', 'temp': 'float32', 'rainfall': 'float32', 'humidity': 'float32'}
)
weather_hourly['date'] = pd.to_datetime(weather_hourly['date'])
weather_hourly['hour'] = weather_hourly['hour'].astype(int)
weather_hourly['rainfall'] = weather_hourly['rainfall'].fillna(0)
print(f'weather_hourly: {len(weather_hourly)}행')

# 캘린더
calendar_df = pd.read_csv(
    os.path.join(EXT_DIR, 'calendar_2018_2026.csv'),
    encoding='utf-8',
    usecols=['date', 'day_of_week', 'is_weekend', 'is_holiday', 'is_non_working'],
    dtype={'day_of_week': 'int8', 'is_weekend': 'int8',
           'is_holiday': 'int8', 'is_non_working': 'int8'}
)
calendar_df['date'] = pd.to_datetime(calendar_df['date'])
print(f'calendar: {len(calendar_df)}행')

print_mem()

## 2. 택시 데이터 청크 처리 - 일별/시간대별/지역별 수요 집계

In [ ]:
# 택시 데이터를 chunk로 읽으면서 일별, 시간별, 지역별 수요 집계
USECOLS = ['RIDE_DTIME', 'RIDE_A_CD']
DTYPE = {'RIDE_DTIME': 'str', 'RIDE_A_CD': 'str'}
CHUNKSIZE = 1_000_000

# 집계용 딕셔너리
daily_counts = {}       # date -> count
hourly_counts = {}      # (date, hour) -> count
region_daily = {}       # (date, RIDE_A_CD) -> count
region_hourly = {}      # (date, hour, RIDE_A_CD) -> count

total_rows = 0
for i, chunk in enumerate(pd.read_csv(
    TAXI_FILE, chunksize=CHUNKSIZE, usecols=USECOLS, dtype=DTYPE
)):
    # 날짜/시간 파싱 (RIDE_DTIME: YYYYMMDDHHmmss 형식)
    chunk['date'] = chunk['RIDE_DTIME'].str[:8]
    chunk['hour'] = chunk['RIDE_DTIME'].str[8:10].astype(int)
    
    # 일별 집계
    dc = chunk.groupby('date').size()
    for d, c in dc.items():
        daily_counts[d] = daily_counts.get(d, 0) + c
    
    # 시간별 집계
    hc = chunk.groupby(['date', 'hour']).size()
    for (d, h), c in hc.items():
        key = (d, h)
        hourly_counts[key] = hourly_counts.get(key, 0) + c
    
    # 지역-일별 집계
    rc = chunk.groupby(['date', 'RIDE_A_CD']).size()
    for (d, r), c in rc.items():
        key = (d, r)
        region_daily[key] = region_daily.get(key, 0) + c
    
    total_rows += len(chunk)
    if (i + 1) % 5 == 0:
        print(f'  {total_rows:,}행 처리 완료')
        print_mem()

print(f'총 {total_rows:,}행 처리 완료')
print_mem()

In [ ]:
# DataFrame으로 변환
df_daily = pd.DataFrame(
    [(k, v) for k, v in daily_counts.items()],
    columns=['date_str', 'trip_count']
)
df_daily['date'] = pd.to_datetime(df_daily['date_str'], format='%Y%m%d')
df_daily = df_daily.drop(columns='date_str').sort_values('date').reset_index(drop=True)

df_hourly = pd.DataFrame(
    [(k[0], k[1], v) for k, v in hourly_counts.items()],
    columns=['date_str', 'hour', 'trip_count']
)
df_hourly['date'] = pd.to_datetime(df_hourly['date_str'], format='%Y%m%d')
df_hourly = df_hourly.drop(columns='date_str').sort_values(['date', 'hour']).reset_index(drop=True)

df_region = pd.DataFrame(
    [(k[0], k[1], v) for k, v in region_daily.items()],
    columns=['date_str', 'region_cd', 'trip_count']
)
df_region['date'] = pd.to_datetime(df_region['date_str'], format='%Y%m%d')
df_region = df_region.drop(columns='date_str').sort_values(['date', 'region_cd']).reset_index(drop=True)

# 메모리 정리
del daily_counts, hourly_counts, region_daily, region_hourly
gc.collect()

print(f'df_daily: {len(df_daily)}행, df_hourly: {len(df_hourly)}행, df_region: {len(df_region)}행')
print_mem()

## 3. 외부 데이터 조인

In [ ]:
# 일별 택시 수요 + 날씨 + 캘린더 조인
merged_daily = df_daily.merge(weather_daily, on='date', how='left') \
                       .merge(calendar_df, on='date', how='left')

# 시간별 택시 수요 + 시간별 날씨 조인
merged_hourly = df_hourly.merge(weather_hourly, on=['date', 'hour'], how='left') \
                         .merge(calendar_df, on='date', how='left')

# 지역별 일 수요 + 날씨 + 캘린더 조인
merged_region = df_region.merge(weather_daily, on='date', how='left') \
                         .merge(calendar_df, on='date', how='left')

print(f'merged_daily: {len(merged_daily)}행')
print(f'merged_hourly: {len(merged_hourly)}행')
print(f'merged_region: {len(merged_region)}행')
merged_daily.head()

## 4. 기온 구간별 택시 수요 분석

In [ ]:
# 기온 구간 정의
temp_bins = [-20, -10, -5, 0, 5, 10, 15, 20, 25, 30, 40]
temp_labels = ['-20~-10', '-10~-5', '-5~0', '0~5', '5~10',
               '10~15', '15~20', '20~25', '25~30', '30~40']

merged_daily['temp_bin'] = pd.cut(
    merged_daily['avg_temp'], bins=temp_bins, labels=temp_labels, right=False
)

# 요일 효과 제거: 비영업일 여부별 평균으로 나누어 정규화
# 먼저 is_non_working별 평균 수요
base_demand = merged_daily.groupby('is_non_working')['trip_count'].mean()
merged_daily['norm_demand'] = merged_daily.apply(
    lambda r: r['trip_count'] / base_demand.get(r['is_non_working'], 1), axis=1
)

# 기온 구간별 수요 통계
temp_demand = merged_daily.groupby('temp_bin', observed=True).agg(
    mean_trips=('trip_count', 'mean'),
    std_trips=('trip_count', 'std'),
    norm_mean=('norm_demand', 'mean'),
    count=('trip_count', 'count')
).round(1)

print('기온 구간별 택시 수요:')
temp_demand

In [ ]:
# 시각화: 기온-수요 산점도
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# 산점도 (평일/비영업일 색상 구분)
colors = merged_daily['is_non_working'].map({0: '#2196F3', 1: '#FF5722'})
axes[0].scatter(merged_daily['avg_temp'], merged_daily['trip_count'],
                c=colors, alpha=0.3, s=10)
# 추세선 (numpy polyfit)
mask = merged_daily['avg_temp'].notna()
z = np.polyfit(merged_daily.loc[mask, 'avg_temp'], merged_daily.loc[mask, 'trip_count'], 3)
p = np.poly1d(z)
x_line = np.linspace(merged_daily['avg_temp'].min(), merged_daily['avg_temp'].max(), 100)
axes[0].plot(x_line, p(x_line), 'k-', linewidth=2, label='추세선(3차)')
axes[0].set_xlabel('평균기온')
axes[0].set_ylabel('일 택시 수요')
axes[0].set_title('기온 vs 택시 수요 (파랑=평일, 주황=비영업일)')
axes[0].legend()

# 기온 구간별 박스플롯
temp_groups = []
temp_group_labels = []
for label in temp_labels:
    subset = merged_daily[merged_daily['temp_bin'] == label]['trip_count']
    if len(subset) > 0:
        temp_groups.append(subset.values)
        temp_group_labels.append(label)

bp = axes[1].boxplot(temp_groups, labels=temp_group_labels, patch_artist=True)
for patch in bp['boxes']:
    patch.set_facecolor('none')
    patch.set_edgecolor('#1976D2')
axes[1].set_xlabel('기온 구간')
axes[1].set_ylabel('일 택시 수요')
axes[1].set_title('기온 구간별 택시 수요 분포')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

## 5. 강수량 구간별 택시 수요 분석

In [ ]:
# 강수량 구간 정의
rain_bins = [-0.1, 0, 1, 5, 10, 30, 100, 500]
rain_labels = ['무강수', '0~1mm', '1~5mm', '5~10mm', '10~30mm', '30~100mm', '100mm+']

merged_daily['rain_bin'] = pd.cut(
    merged_daily['rainfall'], bins=rain_bins, labels=rain_labels, right=True
)

# 강수량 구간별 수요 통계
rain_demand = merged_daily.groupby('rain_bin', observed=True).agg(
    mean_trips=('trip_count', 'mean'),
    std_trips=('trip_count', 'std'),
    norm_mean=('norm_demand', 'mean'),
    count=('trip_count', 'count')
).round(1)

print('강수량 구간별 택시 수요:')
rain_demand

In [ ]:
# 시각화: 강수량-수요 박스플롯
fig, ax = plt.subplots(figsize=(10, 6))

rain_groups = []
rain_group_labels = []
for label in rain_labels:
    subset = merged_daily[merged_daily['rain_bin'] == label]['trip_count']
    if len(subset) > 0:
        rain_groups.append(subset.values)
        rain_group_labels.append(f'{label}\n(n={len(subset)})')

bp = ax.boxplot(rain_groups, labels=rain_group_labels, patch_artist=True)
for patch in bp['boxes']:
    patch.set_facecolor('none')
    patch.set_edgecolor('#E65100')
ax.set_xlabel('강수량 구간')
ax.set_ylabel('일 택시 수요')
ax.set_title('강수량 구간별 택시 수요 분포')
plt.tight_layout()
plt.show()

## 6. 탄력성 계수 산출

탄력성 계수 = (수요변화율) / (기상변화율)

- 기온 탄력성: 기온 1도 변화에 따른 수요 % 변화
- 강수 탄력성: 강수량 1mm 변화에 따른 수요 % 변화

In [ ]:
from scipy import stats

# 평일만 필터 (요일 효과 제거)
weekday_df = merged_daily[merged_daily['is_non_working'] == 0].copy()

# === 기온 탄력성 ===
# log-log 회귀 대신 mid-point elasticity 방식
# 구간별 평균 수요로 arc elasticity 계산
temp_stats = weekday_df.groupby('temp_bin', observed=True).agg(
    avg_demand=('trip_count', 'mean'),
    avg_temp=('avg_temp', 'mean')
).dropna()

# 인접 구간 간 탄력성
elasticity_temp = []
for i in range(1, len(temp_stats)):
    d1, d2 = temp_stats.iloc[i-1]['avg_demand'], temp_stats.iloc[i]['avg_demand']
    t1, t2 = temp_stats.iloc[i-1]['avg_temp'], temp_stats.iloc[i]['avg_temp']
    mid_d = (d1 + d2) / 2
    mid_t = (t1 + t2) / 2
    if mid_t != 0 and mid_d != 0:
        e = ((d2 - d1) / mid_d) / ((t2 - t1) / mid_t)
        elasticity_temp.append({
            'from': temp_stats.index[i-1],
            'to': temp_stats.index[i],
            'temp_change': round(t2 - t1, 1),
            'demand_change_pct': round((d2 - d1) / mid_d * 100, 2),
            'elasticity': round(e, 4)
        })

df_elast_temp = pd.DataFrame(elasticity_temp)
print('기온 탄력성 (구간 간 arc elasticity):')
df_elast_temp

In [ ]:
# === 강수 탄력성 ===
# 비 오는 날 vs 안 오는 날 비교
no_rain = weekday_df[weekday_df['rainfall'] == 0]['trip_count'].mean()

rain_stats = weekday_df[weekday_df['rainfall'] > 0].copy()
rain_stats['rain_bin'] = pd.cut(
    rain_stats['rainfall'],
    bins=[0, 1, 5, 10, 30, 100, 500],
    labels=['0~1', '1~5', '5~10', '10~30', '30~100', '100+']
)

rain_elast = rain_stats.groupby('rain_bin', observed=True).agg(
    avg_demand=('trip_count', 'mean'),
    avg_rainfall=('rainfall', 'mean'),
    count=('trip_count', 'count')
)

rain_elast['demand_change_pct'] = ((rain_elast['avg_demand'] - no_rain) / no_rain * 100).round(2)
print(f'무강수일 평균 수요: {no_rain:,.0f}')
print('\n강수량 구간별 수요 변화율 (평일 기준):')
rain_elast

## 7. 기온 x 강수량 x 시간대 3차원 히트맵

In [ ]:
# 시간별 데이터에서 기온/강수 구간 생성
temp_bins_h = [-20, 0, 10, 20, 30, 40]
temp_labels_h = ['~0', '0~10', '10~20', '20~30', '30~']

rain_bins_h = [-0.1, 0, 5, 30, 500]
rain_labels_h = ['무강수', '약한비(~5)', '보통비(5~30)', '강한비(30~)']

merged_hourly['temp_bin'] = pd.cut(
    merged_hourly['temp'], bins=temp_bins_h, labels=temp_labels_h, right=False
)
merged_hourly['rain_bin'] = pd.cut(
    merged_hourly['rainfall'].fillna(0), bins=rain_bins_h, labels=rain_labels_h, right=True
)

# 기온 x 강수 x 시간대 평균 수요
heatmap_data = merged_hourly.groupby(
    ['temp_bin', 'rain_bin', 'hour'], observed=True
)['trip_count'].mean().reset_index()
heatmap_data.columns = ['temp_bin', 'rain_bin', 'hour', 'avg_demand']

print(f'히트맵 데이터: {len(heatmap_data)}행')
heatmap_data.head(10)

In [ ]:
# 강수 조건별 기온 x 시간 히트맵 (2x2 서브플롯)
fig, axes = plt.subplots(2, 2, figsize=(18, 14))
axes = axes.flatten()

for idx, rain_cat in enumerate(rain_labels_h):
    subset = heatmap_data[heatmap_data['rain_bin'] == rain_cat]
    if len(subset) == 0:
        axes[idx].set_title(f'{rain_cat} (데이터 없음)')
        continue
    
    pivot = subset.pivot_table(
        index='temp_bin', columns='hour', values='avg_demand', aggfunc='mean'
    )
    
    im = axes[idx].imshow(pivot.values, aspect='auto', cmap='YlOrRd')
    axes[idx].set_yticks(range(len(pivot.index)))
    axes[idx].set_yticklabels(pivot.index)
    axes[idx].set_xticks(range(0, len(pivot.columns), 2))
    axes[idx].set_xticklabels([f'{h}시' for h in pivot.columns[::2]])
    axes[idx].set_title(f'강수: {rain_cat}')
    axes[idx].set_xlabel('시간대')
    axes[idx].set_ylabel('기온 구간')
    plt.colorbar(im, ax=axes[idx], label='평균 수요')

fig.suptitle('기온 x 시간대 택시 수요 히트맵 (강수 조건별)', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Plotly 3D 히트맵 (인터랙티브)
hm3d = heatmap_data.copy()
hm3d['temp_num'] = hm3d['temp_bin'].cat.codes
hm3d['rain_num'] = hm3d['rain_bin'].cat.codes

fig = px.scatter_3d(
    hm3d, x='hour', y='temp_num', z='avg_demand',
    color='rain_bin', size='avg_demand',
    labels={'hour': '시간', 'temp_num': '기온구간', 'avg_demand': '평균수요', 'rain_bin': '강수'},
    title='기온 x 강수 x 시간대 택시 수요 3D 시각화'
)
fig.update_layout(scene=dict(
    yaxis=dict(ticktext=temp_labels_h, tickvals=list(range(len(temp_labels_h))))
))
fig.show()

## 8. 지역별(행정동) 날씨 민감도 차이 분석

In [ ]:
# 주요 지역(상위 20개) 선정
top_regions = merged_region.groupby('region_cd')['trip_count'].sum().nlargest(20).index.tolist()
region_top = merged_region[merged_region['region_cd'].isin(top_regions)].copy()

# 비영업일 제외 (평일만)
region_weekday = region_top[region_top['is_non_working'] == 0].copy()

# 강수 민감도: 비 오는 날 vs 안 오는 날 수요 변화율
rain_sensitivity = []
for region in top_regions:
    rdf = region_weekday[region_weekday['region_cd'] == region]
    no_rain_avg = rdf[rdf['rainfall'] == 0]['trip_count'].mean()
    rain_avg = rdf[rdf['rainfall'] > 5]['trip_count'].mean()  # 5mm 이상
    if no_rain_avg > 0 and not np.isnan(rain_avg):
        change = (rain_avg - no_rain_avg) / no_rain_avg * 100
        rain_sensitivity.append({
            'region_cd': region,
            'no_rain_avg': round(no_rain_avg, 1),
            'rain_avg': round(rain_avg, 1),
            'change_pct': round(change, 2)
        })

df_rain_sens = pd.DataFrame(rain_sensitivity).sort_values('change_pct', ascending=False)
print('지역별 강수 민감도 (비 5mm+ 시 수요 변화율, 평일):')
df_rain_sens

In [ ]:
# 기온 민감도: 한파(0도 미만) vs 쾌적(15~25도) 수요 변화율
temp_sensitivity = []
for region in top_regions:
    rdf = region_weekday[region_weekday['region_cd'] == region]
    comfort = rdf[(rdf['avg_temp'] >= 15) & (rdf['avg_temp'] < 25)]['trip_count'].mean()
    cold = rdf[rdf['avg_temp'] < 0]['trip_count'].mean()
    hot = rdf[rdf['avg_temp'] >= 30]['trip_count'].mean()
    if comfort > 0:
        temp_sensitivity.append({
            'region_cd': region,
            'comfort_avg': round(comfort, 1),
            'cold_change_pct': round((cold - comfort) / comfort * 100, 2) if not np.isnan(cold) else None,
            'hot_change_pct': round((hot - comfort) / comfort * 100, 2) if not np.isnan(hot) else None
        })

df_temp_sens = pd.DataFrame(temp_sensitivity)
print('지역별 기온 민감도 (쾌적온도 대비 변화율, 평일):')
df_temp_sens

In [ ]:
# 시각화: 지역별 민감도 비교
fig, axes = plt.subplots(1, 2, figsize=(16, 8))

# 강수 민감도
axes[0].barh(df_rain_sens['region_cd'], df_rain_sens['change_pct'],
             color='none', edgecolor='#1565C0', linewidth=1.5)
axes[0].axvline(x=0, color='gray', linestyle='--', linewidth=0.8)
axes[0].set_xlabel('강수시 수요 변화율 (%)')
axes[0].set_title('지역별 강수 민감도 (비 5mm+ vs 무강수, 평일)')
axes[0].invert_yaxis()

# 기온 민감도 (한파)
df_ts = df_temp_sens.dropna(subset=['cold_change_pct']).sort_values('cold_change_pct')
axes[1].barh(df_ts['region_cd'], df_ts['cold_change_pct'],
             color='none', edgecolor='#C62828', linewidth=1.5)
axes[1].axvline(x=0, color='gray', linestyle='--', linewidth=0.8)
axes[1].set_xlabel('한파시 수요 변화율 (%)')
axes[1].set_title('지역별 한파 민감도 (0도 미만 vs 쾌적, 평일)')
axes[1].invert_yaxis()

plt.tight_layout()
plt.show()

## 9. 결과 해석

### 기온-수요 관계
- 기온 탄력성 계수를 통해 어떤 기온 구간에서 택시 수요가 가장 민감하게 반응하는지 확인
- 일반적으로 한파(0도 미만)와 폭염(30도 이상)에서 수요 증가 패턴 예상
- 쾌적 온도(15~25도)에서 수요가 상대적으로 낮은 U자형 패턴 가능

### 강수량-수요 관계
- 비가 오면 택시 수요가 증가하는 것이 일반적
- 강수량이 많을수록 수요 증가폭이 큰지, 일정 수준 이상에서 오히려 감소하는지 확인
- 무강수 대비 강수시 수요 변화율이 양(+)이면 대체재 효과(도보/대중교통 -> 택시)

### 지역별 민감도
- 강수/기온 민감도가 높은 지역: 도보 이동이 많은 상업지역, 대중교통 접근성이 낮은 지역
- 민감도가 낮은 지역: 이미 택시 이용률이 높거나, 기상과 무관한 업무용 이동 비중이 높은 지역
- 행정동 코드 기준이므로 실제 지역명 매핑 시 더 구체적 해석 가능

### 3차원 히트맵 (기온 x 강수 x 시간대)
- 출퇴근 시간대(7~9시, 17~19시)에 기상 영향이 더 크게 나타나는지 확인
- 야간 시간대(22~02시)에 비가 오면 귀가 택시 수요 급증 여부
- 기온과 강수의 복합 효과(춥고 비 오는 날)가 개별 효과보다 큰지 시너지 분석

In [ ]:
# 메모리 정리
del merged_daily, merged_hourly, merged_region, region_weekday
del df_daily, df_hourly, df_region
gc.collect()
print('분석 완료')
print_mem()

## 10. [보강] 시계열 추세

In [ ]:
# === [시계열 보강] 일별 수요 추세 (7·30일 이동평균) ===
# 기존 분석과 독립적으로 일별 시계열을 다시 집계해 장기 추세를 확인한다.
import pandas as _pd, numpy as _np, matplotlib.pyplot as _plt
_daily = {}
for _ck in _pd.read_csv(TAXI_FILE if 'TAXI_FILE' in dir() else './DC_TBYXD012.csv',
                        usecols=['RIDE_DTIME'], dtype={'RIDE_DTIME': str}, chunksize=1_000_000):
    _d = _ck['RIDE_DTIME'].str[:8]
    _d = _d[_d.str.match(r'\d{8}')]
    for _k, _v in _d.groupby(_d).size().items():
        _daily[_k] = _daily.get(_k, 0) + _v
    del _ck
_ts = _pd.Series(_daily); _ts.index = _pd.to_datetime(_ts.index, format='%Y%m%d')
_ts = _ts.sort_index().asfreq('D').interpolate()
_ma7, _ma30 = _ts.rolling(7, center=True).mean(), _ts.rolling(30, center=True).mean()
fig, ax = _plt.subplots(figsize=(18, 5))
ax.plot(_ts.index, _ts.values, lw=0.3, alpha=0.4, color='gray', label='일별')
ax.plot(_ma7.index, _ma7.values, lw=1.2, color='steelblue', label='7일 이동평균')
ax.plot(_ma30.index, _ma30.values, lw=2, color='darkorange', label='30일 이동평균')
ax.set_title('일별 택시 수요 추세 (7·30일 이동평균)', fontweight='bold')
ax.set_xlabel('날짜'); ax.set_ylabel('일 건수'); ax.legend(); ax.grid(alpha=0.3)
_plt.tight_layout(); _plt.show()
print(f"기간 {_ts.index.min().date()} ~ {_ts.index.max().date()}, 일평균 {_ts.mean():,.0f}건")